# House Prices — Baseline Model

This notebook is the **baseline reference** for Mini Project 1.

Goal:
1. Load the raw House Prices training data.
2. Split into train/validation **before fitting preprocessing** to avoid data leakage.
3. Build a reproducible preprocessing + regression pipeline.
4. Evaluate with **MAE and RMSE**.
5. Save the complete fitted pipeline to `models/baseline.pkl`.

This notebook will later be refactored into an OOP Python package. The baseline metrics should remain approximately stable after refactoring.


In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TARGET = "SalePrice"

DATA_PATH = Path("../data/raw/train.csv")
MODEL_PATH = Path("../models/baseline.pkl")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Put train.csv inside data/raw/."
    )


In [ ]:
# Load raw data
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())


In [ ]:
# Basic validation
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found.")

print("Missing target values:", df[TARGET].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

display(df.describe(include="all").T.head(20))


## Train / validation split

We split **before** learning imputers, encoders, scalers, or any other preprocessing statistics. This prevents information from the validation set from leaking into training.


In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Id is an identifier, not a useful predictive feature.
if "Id" in X.columns:
    X = X.drop(columns=["Id"])

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)


In [ ]:
# Detect feature types from the training split only
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


## Preprocessing

For the baseline we intentionally keep preprocessing simple and reproducible:

- Numerical features → median imputation → standard scaling
- Categorical features → most-frequent imputation → one-hot encoding

The preprocessing is part of the sklearn `Pipeline`, so the exact same transformations are automatically applied during prediction.


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)


## Train baseline model

In [ ]:
baseline_model.fit(X_train, y_train)
print("Baseline model trained successfully.")


## Validation metrics

We report both metrics required for the baseline:

- **MAE** — average absolute prediction error.
- **RMSE** — penalizes larger errors more strongly.


In [ ]:
y_pred = baseline_model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"Validation MAE:  {mae:,.2f}")
print(f"Validation RMSE: {rmse:,.2f}")


In [ ]:
# Save the COMPLETE fitted pipeline.
# This includes preprocessing + trained regression model.
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

with MODEL_PATH.open("wb") as f:
    pickle.dump(baseline_model, f)

print(f"Saved baseline model to: {MODEL_PATH.resolve()}")


## Reload test

The saved artifact must be usable independently of the training cell.


In [ ]:
with MODEL_PATH.open("rb") as f:
    loaded_model = pickle.load(f)

reloaded_predictions = loaded_model.predict(X_val)

reloaded_mae = mean_absolute_error(y_val, reloaded_predictions)

print(f"Reloaded model MAE: {reloaded_mae:,.2f}")
assert np.isclose(mae, reloaded_mae), "Reloaded model changed the prediction results."

print("Model save/load check passed.")


## Baseline record

Record these two numbers in the project README once the notebook has been executed:

- Validation MAE
- Validation RMSE

These values become the reference point for the later OOP refactoring.
